In [ ]:
import numpy as np
import matplotlib.pyplot as plt

g = 9.81
L = 1.0
omega0 = np.sqrt(g/L)

theta0 = 0.1
thetadot0 = 0.0

T = 10.0
dt = 0.01

print(f'g={g}, L={L}, theta0={theta0}, thetadot0={thetadot0}, T={T}, dt={dt}')

In [ ]:
def f(y, t, g=9.81, L=1.0):
    theta, omega = y
    return np.array([omega, -(g/L)*np.sin(theta)], dtype=float)

def step_euler(y, t, dt, g=9.81, L=1.0):
    return y + dt * f(y, t, g=g, L=L)

def step_rk4(y, t, dt, g=9.81, L=1.0):
    k1 = f(y, t, g=g, L=L)
    k2 = f(y + 0.5*dt*k1, t + 0.5*dt, g=g, L=L)
    k3 = f(y + 0.5*dt*k2, t + 0.5*dt, g=g, L=L)
    k4 = f(y + dt*k3, t + dt, g=g, L=L)
    return y + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def simulate(stepper, dt, T, theta0, thetadot0, g=9.81, L=1.0):
    N = int(np.round(T/dt)) + 1
    t = np.linspace(0.0, dt*(N-1), N)
    y = np.zeros((N, 2), dtype=float)
    y[0] = np.array([theta0, thetadot0], dtype=float)
    for i in range(N-1):
        y[i+1] = stepper(y[i], t[i], dt, g=g, L=L)
    return t, y[:,0], y[:,1]

In [ ]:
def analytic_small_angle_theta(t, theta0, thetadot0, g=9.81, L=1.0):
    w = np.sqrt(g/L)
    A = theta0
    B = thetadot0 / w
    return A*np.cos(w*t) + B*np.sin(w*t)

def analytic_small_angle_omega(t, theta0, thetadot0, g=9.81, L=1.0):
    w = np.sqrt(g/L)
    A = theta0
    B = thetadot0 / w
    return -A*w*np.sin(w*t) + B*w*np.cos(w*t)

In [ ]:
def energy_full(theta, omega, g=9.81, L=1.0, m=1.0):
    return 0.5*m*(L**2)*omega**2 + m*g*L*(1.0 - np.cos(theta))

def energy_sho(theta, omega, g=9.81, L=1.0, m=1.0):
    return 0.5*m*(L**2)*omega**2 + 0.5*m*g*L*theta**2

In [ ]:
t, th_e, om_e = simulate(step_euler, dt, T, theta0, thetadot0, g=g, L=L)
_, th_r, om_r = simulate(step_rk4, dt, T, theta0, thetadot0, g=g, L=L)

th_a = analytic_small_angle_theta(t, theta0, thetadot0, g=g, L=L)
om_a = analytic_small_angle_omega(t, theta0, thetadot0, g=g, L=L)

E_e = energy_full(th_e, om_e, g=g, L=L)
E_r = energy_full(th_r, om_r, g=g, L=L)
E_a = energy_sho(th_a, om_a, g=g, L=L)

E0_e, E0_r, E0_a = E_e[0], E_r[0], E_a[0]

fig, axs = plt.subplots(2, 2, figsize=(12, 8))

# theta(t)
ax = axs[0, 0]
ax.plot(t, th_a, label='аналитическое')
ax.plot(t, th_e, '--', label='Эйлер')
ax.plot(t, th_r, ':', label='RK4')
ax.set_xlabel('t, с'); ax.set_ylabel(r'$\theta$, рад'); ax.grid(True); ax.legend()

# фазовый портрет
ax = axs[0, 1]
ax.plot(th_a, om_a, label='аналитическое')
ax.plot(th_e, om_e, '--', label='Эйлер')
ax.plot(th_r, om_r, ':', label='RK4')
ax.set_xlabel(r'$\theta$, рад'); ax.set_ylabel(r'$\omega$, рад/с'); ax.grid(True); ax.legend()

# энергия
ax = axs[1, 0]
ax.plot(t, (E_e - E0_e)/E0_e, '--', label='Эйлер ΔE/E0')
ax.plot(t, (E_r - E0_r)/E0_r, ':', label='RK4 ΔE/E0')
ax.hlines(0.0, t[0], t[-1], linestyles='-', alpha=0.6, label='аналитическое')
ax.set_xlabel('t, с'); ax.set_ylabel('Относит. ошибка энергии'); ax.grid(True); ax.legend()

# ошибки координаты
ax = axs[1, 1]
err_e = np.abs(th_e - th_a); err_r = np.abs(th_r - th_a)
ax.plot(t, err_e, '--', label='|θ−θ_ан| Эйлер')
ax.plot(t, err_r, ':', label='|θ−θ_ан| RK4')
ax.set_xlabel('t, с'); ax.set_ylabel('Ошибка угла, рад'); ax.grid(True); ax.legend()

plt.tight_layout()
plt.show()

# Оценка линейной скорости накопления ошибок
slope_e, _ = np.polyfit(t, err_e, 1)
slope_r, _ = np.polyfit(t, err_r, 1)
slope_E_e, _ = np.polyfit(t, np.abs((E_e - E0_e)/E0_e), 1)
slope_E_r, _ = np.polyfit(t, np.abs((E_r - E0_r)/E0_r), 1)

print(f'Скорость роста ошибки угла: Эйлер ≈ {slope_e:.3e} рад/с, RK4 ≈ {slope_r:.3e} рад/с')
print(f'Скорость роста ошибки энергии: Эйлер ≈ {slope_E_e:.3e} 1/с, RK4 ≈ {slope_E_r:.3e} 1/с')

In [ ]:
def plot_phase_portraits(theta0_list, thetadot0=0.0, dt=0.01, T=10.0, method='rk4', g=9.81, L=1.0):
    stepper = step_rk4 if method.lower() == 'rk4' else step_euler
    fig, ax = plt.subplots(figsize=(6, 5))
    for th0 in theta0_list:
        t, th, om = simulate(stepper, dt, T, th0, thetadot0, g=g, L=L)
        ax.plot(th, om, label=fr'$\theta_0={th0:.3f}$')
    ax.set_xlabel(r'$\theta$, рад'); ax.set_ylabel(r'$\omega$, рад/с')
    ax.grid(True); ax.legend()
    plt.show()


theta0_list = [0.05, 0.2, 0.5, 1.0, 1.5]
plot_phase_portraits(theta0_list, thetadot0=0.0, dt=dt, T=T, method='rk4', g=g, L=L)

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

x_r = L*np.sin(th_r)
y_r = -L*np.cos(th_r)

fig, ax = plt.subplots(figsize=(5, 5))
ax.set_aspect('equal')
ax.set_xlim(-1.2*L, 1.2*L)
ax.set_ylim(-1.2*L, 0.2*L)
ax.grid(True)

line, = ax.plot([], [], lw=2)
bob, = ax.plot([], [], 'o', ms=8)

def init():
    line.set_data([], [])
    bob.set_data([], [])
    return line, bob

def update(i):
    x = x_r[i]; y = y_r[i]
    line.set_data([0.0, x], [0.0, y])
    bob.set_data([x], [y])
    return line, bob

ani = FuncAnimation(fig, update, frames=len(t), init_func=init, interval=1000*dt, blit=True)
HTML(ani.to_jshtml())

In [ ]:
def lengths_from_frequency_ratios(p_list, L_base=1.0):
    p0 = p_list[0]
    return np.array([L_base*(p0/float(p))**2 for p in p_list], dtype=float)

def simulate_pendulum_L(Li, theta0, thetadot0, dt, T, g=9.81):
    def fL(y, t):
        th, om = y
        return np.array([om, -(g/Li)*np.sin(th)], dtype=float)
    def rk4L(y, t, dt):
        k1 = fL(y, t)
        k2 = fL(y + 0.5*dt*k1, t + 0.5*dt)
        k3 = fL(y + 0.5*dt*k2, t + 0.5*dt)
        k4 = fL(y + dt*k3, t + dt)
        return y + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)
    N = int(np.round(T/dt)) + 1
    t = np.linspace(0.0, dt*(N-1), N)
    y = np.zeros((N, 2), dtype=float)
    y[0] = np.array([theta0, thetadot0], dtype=float)
    for i in range(N-1):
        y[i+1] = rk4L(y[i], t[i], dt)
    th = y[:,0]; om = y[:,1]
    x = Li*np.sin(th); y_cart = -Li*np.cos(th)
    return t, th, om, x, y_cart

p_list = [1, 2, 3, 4]
L_list = lengths_from_frequency_ratios(p_list, L_base=L)

theta0_multi = 0.1
thetadot0_multi = 0.0

sols = [simulate_pendulum_L(Li, theta0_multi, thetadot0_multi, dt, T, g=g) for Li in L_list]

# Визуализация множества: θ(t) и фазовые портреты
fig, axs = plt.subplots(1, 2, figsize=(12, 4))
for Li, sol in zip(L_list, sols):
    t_i, th_i, om_i, x_i, y_i = sol
    axs[0].plot(t_i, th_i, label=f'L={Li:.3f}')
    axs[1].plot(th_i, om_i, label=f'L={Li:.3f}')
axs[0].set_xlabel('t, с'); axs[0].set_ylabel(r'$\theta$, рад'); axs[0].grid(True); axs[0].legend()
axs[1].set_xlabel(r'$\theta$, рад'); axs[1].set_ylabel(r'$\omega$, рад/с'); axs[1].grid(True); axs[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_aspect('equal')
R = 1.2*np.max(L_list)
ax.set_xlim(-R, R)
ax.set_ylim(-R, 0.2*R)
ax.grid(True)

lines = []; bobs = []
for _ in L_list:
    ln, = ax.plot([], [], lw=2)
    pt, = ax.plot([], [], 'o', ms=6)
    lines.append(ln); bobs.append(pt)

def init_multi():
    for ln, pt in zip(lines, bobs):
        ln.set_data([], [])
        pt.set_data([], [])
    return (*lines, *bobs)

def update_multi(i):
    for idx, sol in enumerate(sols):
        _, _, _, x_i, y_i = sol
        lines[idx].set_data([0.0, x_i[i]], [0.0, y_i[i]])
        bobs[idx].set_data([x_i[i]], [y_i[i]])
    return (*lines, *bobs)

ani_multi = FuncAnimation(fig, update_multi, frames=len(t), init_func=init_multi, interval=1000*dt, blit=True)
HTML(ani_multi.to_jshtml())